# ప్రాంప్ట్ ఇంజనీరింగ్‌కు పరిచయం
ప్రాంప్ట్ ఇంజనీరింగ్ అనేది సహజ భాషా ప్రాసెసింగ్ పనులకు ప్రాంప్ట్‌లను రూపకల్పన మరియు మెరుగుపరచే ప్రక్రియ. ఇది సరైన ప్రాంప్ట్‌లను ఎంపిక చేసుకోవడం, వాటి పారామితులు సవరించడం మరియు వాటి పనితీరును మూల్యాంకనం చేయడం involve చేస్తుంది. NLP మోడల్స్‌లో అధిక ఖచ్చితత్వం మరియు సామర్థ్యాన్ని సాధించడానికి ప్రాంప్ట్ ఇంజనీరింగ్ అవసరం. ఈ విభాగంలో, OpenAI మోడల్స్ ఉపయోగించి ప్రాంప్ట్ ఇంజనీరింగ్ యొక్క మౌలికాలను పరిశీలిస్తాం.


### వ్యాయామం 1: టోకనైజేషన్
OpenAI నుండి ఒక ఓపెన్-సోర్స్ వేగవంతమైన టోకనైజర్ అయిన tiktoken ఉపయోగించి టోకనైజేషన్‌ని అన్వేషించండి
మరిన్ని ఉదాహరణల కోసం [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) ని చూడండి.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### వ్యాయామం 2: Microsoft Foundry మోడల్స్ కీ సెటప్ ధ్రువీకరణ  

> **గమనిక:** GitHub మోడల్స్ జూలై 2026 చివరగా రిటైర్ అవుతోంది. ఈ వ్యాయామం [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) ఉపయోగిస్తుంది, ఇది ఉచితంగా ప్రయత్నించగల మోడల్ క్యాటలాగ్ మరియు Azure AI Inference SDK అనుభవాన్ని అందిస్తుంది.  

కింది కోడ్ ను నడపండి మరియు మీ Microsoft Foundry Models ఎండ్‌పాయింట్ సరైనవిగా సెటప్ అయ్యిందా అని ధృవీకరించండి. కోడ్ ఒక సాదారణ మౌలిక ప్రాంప్ట్ ను ప్రయత్నించి పూర్తి అవుతుందో లేదో తనిఖీ చేస్తుంది. మీరు ఇన్‌పుట్ గా ఇచ్చే `oh say can you see` వాక్యం `by the dawn's early light..` లాంటి ఆఖరి వాక్యంతో పూర్తవాలి.  


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

# temperature/top_p need a non-reasoning model (gpt-5 rejects them), so use a Llama model here.
model_name = os.environ.get("AZURE_INFERENCE_CHAT_MODEL", "Llama-3.3-70B-Instruct")

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

def get_completion(prompt, client, model_name, temperature=1.0, max_tokens=1000, top_p=1.0):
    response = client.complete(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt, client, model_name)
print(response)


That line is the opening lyric of "The Star-Spangled Banner," the national anthem of the United States, written by Francis Scott Key. If you'd like more information or analysis, feel free to ask!


### వ్యాయామం 3: కల్పనలు
మీరు LLMని ప్రాంప్ట్‌పై పూర్తి వివరాలు ఇవ్వమని అడిగినప్పుడు, అది ఉండకపోవచ్చు లేదా అది ముందుగా శిక్షణ పొందిన డేటాసెట్‌కు (ఇంతకాలం తర్వాతి) వెలుపల ఉన్నప్పటికీ తెలియకపోవచ్చు విషయం గురించి ఏం జరుగుతుందో ఆవిష్కరించండి. మీరు వేరే ప్రాంప్ట్ లేదా వేరే మోడల్ ప్రయత్నిస్తే జవాబు ఎలా మారుతుందో చూడండి.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt, client, model_name)
print(response)

### వ్యాయామం 4: సూచన ఆధారిత  
ప్రాథమిక విషయాన్ని సెట్ చేయడానికి "text" చరాన్ని ఉపయోగించండి  
మరియు ఆ ప్రాథమిక విషయం సంబంధించి ఒక సూచన ఇవ్వడానికి "prompt" చరాను ఉపయోగించండి.  

ఇక్కడ మేము మోడల్‌ను రెండో తరగతి విద్యార్థి కోసం టెక్స్ట్‌ను సంక్షిప్తంగా చెప్పమని కోరుకుంటున్నాము  


In [4]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt, client, model_name)
print(response)

Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It's made of gas and is much bigger than all the other planets put together! You can see Jupiter in the night sky because it's very bright. People have noticed it for a really long time and named it after a Roman god.


### వ్యాయామం 5: సంక్లిష్ట ప్రాంప్ట్ 
సిస్టమ్, యూజర్ మరియు అసిస్టెంట్ సందేశాలు కలిగిన అభ్యర్థనను ప్రయత్నించండి 
సిస్టమ్ అసిస్టెంట్ పరిసరాన్ని సెటప్ చేస్తుంది
యూజర్ & అసిస్టెంట్ సందేశాలు బహుళ-తిరుగుడు సంభాషణ పరిసరాన్ని అందిస్తాయి

అసిస్టెంట్ వ్యక్తిత్వం "వినోదాత్మక" గా సిస్టమ్ పరిసరంలో ఎలా సెట్ చేయబడిందో గమనించండి. 
ఒక వేరే వ్యక్తిత్వ పరిసరాన్ని ప్రయత్నించండి. లేక వేరే ఇన్‌పుట్/ఆవుట్‌పుట్ సందేశాల శ్రేణిని ప్రయత్నించండి


In [5]:
response = client.complete(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

Oh, you mean the famous 2020 World Series that wasn’t in a regular location? That was the year they played in the glamorous Arlington, Texas, at Globe Life Field.


### వ్యాయామం: మీ అనుభూతిని అన్వేషించండి
పై ఉదాహరణలు మీరు కొత్త ప్రాంప్ట్‌లు (సాధారణ, సంక్లిష్ట, సూచనాత్మక వంటివి) సృష్టించడానికి ఉపయోగించుకోవచ్చని నమూనాలను అందిస్తాయి - మేము చర్చించిన ఇతర ఆలోచనలు లాంటి ఉదాహరణలు, సూచనలు మరియు మరిన్ని విషయాలను అన్వేషించడానికి ఇతర వ్యాయామాలను సృష్టించడానికి ప్రయత్నించండి.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
